#  Curso: LLM-as-a-judge y evaluación humana

Anteriormente usamos un LLM para verificar afirmaciones (*"¿el contexto soporta esto? sí/no"*).
Eso ya era **LLM-as-a-judge**, sin nombrarlo. Hoy lo formalizamos: cómo diseñar un juez
confiable, cuáles son sus sesgos conocidos, y cómo saber si realmente podemos confiar en él.

### Objetivos de esta clase
- ✅ Entender qué es LLM-as-a-judge y por qué se volvió el estándar de la industria
- ✅ Diseñar **rúbricas** efectivas (la diferencia entre un juez útil y uno ruidoso)
- ✅ Implementar evaluación **pointwise** (score absoluto) y **pairwise** (A vs B)
- ✅ Medir empíricamente los sesgos clásicos: **posición**, **verbosidad**, **auto-preferencia**
- ✅ Calibrar el juez contra anotación humana con **Cohen's Kappa**
- ✅ Conocer las herramientas de la industria (promptfoo y similares)

### La pregunta central de esta clase

> Le pedimos a un LLM que califique una respuesta del 1 al 5... ¿ese número significa
> algo, o es solo una alucinación numérica con forma de score?


## Instalación de Dependencias

In [ ]:
!pip install transformers torch huggingface_hub scikit-learn -q

In [ ]:
HB_TOKEN="TU_TOKEN_AQUI"

## Acceso a Gemma (modelo con licencia)

Si ya ejecutaste `notebook_login()` en una clase anterior de esta sesión de Colab, puedes
saltarte este paso. Si no:

1. Entra a **https://huggingface.co/google/gemma-3-1b-it**, inicia sesión y acepta la licencia.
2. Crea un token en **https://huggingface.co/settings/tokens** (alcanza con *Read*).
3. Corre la celda de abajo y pega tu token.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Importaciones y Setup

Usamos **Gemma 3 1B Instruct**, en un **triple rol** hoy: generador, verificador (como en
la Clase 3) y ahora también **juez con rúbrica**. Es exactamente el patrón que vas a
encontrar en producción: el mismo modelo (o uno más grande) evaluando lo que otros modelos
—o él mismo— produjeron.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import re
import numpy as np
from collections import Counter
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [ ]:
print("🔄 Loading LLM...")
model_name = "google/gemma-3-1b-it"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

🔄 Loading LLM...


config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.00GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [ ]:
llm = pipeline("text-generation", model=model, tokenizer=tokenizer)


def ask_llm(prompt, max_new_tokens=60, do_sample=False, temperature=0.7):
    """Envuelve el pipeline para chatear con Gemma usando su chat template."""
    messages = [{"role": "user", "content": prompt}]
    kwargs = dict(max_new_tokens=max_new_tokens, do_sample=do_sample,
                  pad_token_id=tokenizer.eos_token_id, return_full_text=False)
    if do_sample:
        kwargs["temperature"] = temperature
    output = llm(messages, **kwargs)
    return output[0]["generated_text"].strip()


print("✅ LLM ready")

✅ LLM ready


---
# PARTE 1: ¿Qué es LLM-as-a-judge?

## 1.1 Por qué se volvió el estándar

Evaluar con humanos es el estándar de oro... y es lento y caro. Si cambias un prompt
100 veces por semana, no puedes esperar 3 días de anotación humana por cada cambio.

**LLM-as-a-judge**: usar un LLM (idealmente potente) para evaluar la salida de otro
sistema, siguiendo criterios explícitos — igual que hicimos en la Clase 3 con
`verify_statement` y `judge_chunk_useful`, pero ahora con una **rúbrica completa** en
vez de un simple sí/no.

Ventajas: escala, es barato, es rápido, es consistente *dentro* de una misma corrida.
Riesgos: hereda los sesgos y errores del juez — por eso esta clase existe.

## El ejemplo de hoy: Valentina, la jueza del concurso de dibujos

Imaginemos un concurso de dibujo infantil, y a **Valentina**, la jueza. A lo largo de
esta clase, Valentina va a hacer exactamente lo que le vamos a pedir a Gemma:

- Calificar un dibujo ella sola, con una rúbrica (*pointwise*)
- Comparar dos dibujos lado a lado y decir cuál le gusta más (*pairwise*)
- Y, sin darse cuenta, cometer sesgos muy humanos — los mismos que cometen los LLMs.

Guarda a Valentina en la cabeza — la usamos en cada sección.


---
# PARTE 2: Diseño de rúbricas (evaluación pointwise)

## Concepto: qué hace buena a una rúbrica

Una rúbrica define: **criterios** (¿qué se evalúa?), una **escala** (¿de cuánto a
cuánto?), y **anclas** (¿qué significa cada número, en concreto?).

Compara estos dos prompts para el mismo juez:

**❌ Rúbrica vaga**: *"Rate this answer from 1 to 10."*
Problema: cada corrida puede interpretar la escala distinto — ¿un 7 es "bueno" o
"mediocre"? El juez inventa su propio criterio cada vez, y eso agrega **varianza**
que no viene de la calidad real de la respuesta, sino del ruido del juez.

**✅ Rúbrica con anclas**, evaluando *groundedness* (fidelidad al contexto):

```
1 = Completamente falsa o inventada
2 = Mayormente incorrecta, con algún elemento correcto suelto
3 = Parcialmente correcta, pero le faltan datos clave o tiene errores
4 = Mayormente correcta y completa, con problemas menores
5 = Totalmente correcta, completa, y respaldada por el contexto
```

Con anclas, dos corridas del mismo juez (o dos jueces distintos) tienden a converger
al mismo número — **eso es lo que hace que un score sea medible y no solo vibes**.

### Con Valentina

**Rúbrica vaga**: "¿Te gusta el dibujo? Del 1 al 10." → Valentina puede dar 6 hoy y 9
mañana con el MISMO dibujo, según su humor.

**Rúbrica con anclas**: "1 = no sigue la consigna en absoluto, 3 = sigue la consigna
pero le faltan detalles, 5 = sigue la consigna perfectamente y con detalle" → ahora
Valentina (o cualquier otro jurado) da un número **repetible**, anclado a algo concreto.


In [ ]:
RUBRIC = """Rate the ANSWER on a scale of 1 to 5 for how well it is grounded in the CONTEXT.
Use exactly this scale:
1 = Completely false or fabricated, not supported by the context at all
2 = Mostly incorrect, with at most one correct detail
3 = Partially correct, but missing key information or containing errors
4 = Mostly correct and complete, with minor issues
5 = Fully correct, complete, and fully supported by the context

Respond in EXACTLY this format, with no other text:
Score: X"""

def extract_score(text):
    """
    Extrae el score 1-5. Buscamos primero el formato exacto que pedimos
    ('Score: X'); si el modelo no lo respeta, un fallback busca cualquier
    dígito 1-5 aislado (red de seguridad, no sustituto de un buen prompt).
    """
    m = re.search(r"\bScore:\s*([1-5])\b", text, re.IGNORECASE)
    if m:
        return int(m.group(1))
    m = re.search(r"\b([1-5])\b", text)
    return int(m.group(1)) if m else None

def judge_pointwise(question, answer, context, rubric=RUBRIC):
    prompt = f"{rubric}\n\nContext: {context}\n\nQuestion: {question}\n\nAnswer: {answer}"
    output = ask_llm(prompt, max_new_tokens=10)
    return extract_score(output), output

print("✅ judge_pointwise defined")

✅ judge_pointwise defined


## Probemos con el escenario de CloudBox

Reusamos los 3 chunks recuperados y los 4 casos diseñados (buena / alucinada /
irrelevante / evasiva) de la Clase 3, para ver si el score pointwise los ordena
como esperamos.

In [ ]:
context_chunks = [
    "Free plan users can upload files up to 2GB each. Pro and Team plan users can upload files up to 100GB each.",
    "CloudBox offers a REST API for developers to programmatically upload, list, and manage files, rate-limited to 1000 requests per hour.",
    "Deleted files stay in the Trash for 30 days before being permanently removed. You can restore them anytime during that window.",
]
context = " ".join(context_chunks)
question = "What is the file size limit on the Free plan, and how long do deleted files stay in the Trash?"

cases = {
    "✅ buena": "On the Free plan, files can be up to 2GB each. Deleted files stay in the Trash for 30 days before being permanently removed.",
    "❌ alucinada": "On the Free plan, files can be up to 2GB each. Deleted files stay in the Trash forever, so you never lose anything.",
    "❌ irrelevante": "CloudBox offers a REST API for developers. It is rate-limited to 1000 requests per hour.",
    "❌ evasiva": "CloudBox has many useful features for managing your files.",
}

print("🔬 Scores pointwise:\n")
for label, ans in cases.items():
    score, raw = judge_pointwise(question, ans, context)
    print(f"{label:<15} → Score: {score}   (raw: '{raw}')")

print("""
❓ Discusión: ¿el score ordenó los 4 casos como esperarías? Compáralo con los
   resultados de Faithfulness y Answer Relevancy de la Clase 3 — ¿un solo número
   pointwise puede capturar lo que dos métricas RAGAS distintas miden por separado?
""")

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


🔬 Scores pointwise:



[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ buena         → Score: 5   (raw: 'Score: 5')


[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


❌ alucinada     → Score: 1   (raw: 'Score: 1')


[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


❌ irrelevante   → Score: 2   (raw: 'Score: 2')
❌ evasiva       → Score: 1   (raw: 'Score: 1')

❓ Discusión: ¿el score ordenó los 4 casos como esperarías? Compáralo con los
   resultados de Faithfulness y Answer Relevancy de la Clase 3 — ¿un solo número
   pointwise puede capturar lo que dos métricas RAGAS distintas miden por separado?



---
# PARTE 3: Pairwise vs Pointwise

## Concepto

| | **Pointwise** | **Pairwise** |
|---|---|---|
| ¿Qué produce? | un score absoluto (ej. 1-5) | una preferencia relativa (A, B, o empate) |
| ¿Para qué sirve? | dashboards, monitoreo a lo largo del tiempo | comparar dos prompts, dos modelos, dos versiones |
| ¿Qué tan confiable es? | más ruidoso — la escala absoluta es difícil de anclar | más confiable — comparar es más fácil que puntuar (hallazgo consistente en la literatura de evaluación) |
| ¿Costo? | 1 llamada por respuesta | 1 llamada por *par* — crece más rápido si comparas muchas versiones |

Regla práctica: si vas a decidir *"¿el prompt nuevo es mejor que el viejo?"*, usa
pairwise. Si vas a graficar *"la calidad promedio de las respuestas esta semana"*,
necesitas un número por respuesta — pointwise (aunque también puedes derivar un
número agregando muchas comparaciones pairwise, como el rating Elo de ajedrez —
así funciona Chatbot Arena).

### Con Valentina

**Pointwise**: Valentina mira UN dibujo, sola, y dice "le doy un 4 de 5".

**Pairwise**: Valentina mira DOS dibujos lado a lado y dice "el de la izquierda me
gusta más" — no necesita inventar una escala absoluta, solo comparar. Por eso los
jueces (humanos y LLMs) suelen ser más consistentes comparando que puntuando.


In [ ]:
def parse_pairwise_verdict(text):
    """
    'Tie' se revisa primero porque la palabra 'tie' no comparte letras sueltas
    con A o B. Para A/B usamos \b (límite de palabra) por la misma razón que en
    la Clase 1: sin eso, una palabra como 'Answer' podría confundirse con la letra A.
    """
    low = text.strip().lower()
    if "tie" in low:
        return "Tie"
    m = re.search(r"\b(a|b)\b", low)
    return m.group(1).upper() if m else None

def judge_pairwise(question, answer_a, answer_b, context, max_token=20):
    prompt = (
        f"Context: {context}\n\nQuestion: {question}\n\n"
        f"Answer A: {answer_a}\n\nAnswer B: {answer_b}\n\n"
        f"Which answer is better — more accurate and complete based on the context? "
        f"Respond with exactly one word: A, B, or Tie."
    )
    output = ask_llm(prompt, max_new_tokens=max_token)
    return parse_pairwise_verdict(output)

print("✅ judge_pairwise defined")

✅ judge_pairwise defined


In [ ]:
# Round-robin: comparamos cada par de casos y contamos victorias
labels = list(cases.keys())
wins = {l: 0 for l in labels}

print("🔬 Comparaciones pairwise (round-robin):\n")
for i in range(len(labels)):
    for j in range(i + 1, len(labels)):
        a, b = labels[i], labels[j]
        verdict = judge_pairwise(question, cases[a], cases[b], context)
        if verdict == "A":
            wins[a] += 1; result = f"{a} gana"
        elif verdict == "B":
            wins[b] += 1; result = f"{b} gana"
        else:
            result = "empate"
        print(f"   {a}  vs  {b}  →  {result}")

print("\n📊 Ranking implícito (por victorias):")
for label, w in sorted(wins.items(), key=lambda x: -x[1]):
    print(f"   {label}: {w} victorias")

print("""
❓ ¿El ranking pairwise coincide con el orden que dieron los scores pointwise
   de la Parte 2? Si no coinciden, ¿en qué caso specific difieren, y por qué
   podría pasar?
""")

[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🔬 Comparaciones pairwise (round-robin):



[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   ✅ buena  vs  ❌ alucinada  →  ✅ buena gana


[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

   ✅ buena  vs  ❌ evasiva  →  ✅ buena gana


[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   ❌ alucinada  vs  ❌ irrelevante  →  empate


[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   ❌ alucinada  vs  ❌ evasiva  →  ❌ alucinada gana
   ❌ irrelevante  vs  ❌ evasiva  →  ❌ irrelevante gana

📊 Ranking implícito (por victorias):
   ✅ buena: 3 victorias
   ❌ alucinada: 1 victorias
   ❌ irrelevante: 1 victorias
   ❌ evasiva: 0 victorias

❓ ¿El ranking pairwise coincide con el orden que dieron los scores pointwise
   de la Parte 2? Si no coinciden, ¿en qué caso specific difieren, y por qué
   podría pasar?



---
# PARTE 4: Sesgo de posición (*position bias*)

## Concepto

Los LLM-jueces tienden a preferir sistemáticamente la respuesta que aparece
**primero** (o, en algunos modelos, la que aparece última) — sin importar el
contenido. Es un sesgo bien documentado en la literatura de LLM-as-a-judge.

### Cómo medirlo

Para cada par $(X, Y)$, lo evaluamos **dos veces**, invirtiendo el orden:

$$\text{veredicto}_1 = \text{judge}(A{=}X, B{=}Y) \qquad \text{veredicto}_2 = \text{judge}(A{=}X, B{=}Y \to A{=}Y, B{=}X)$$

Si el juez fuera perfectamente consistente, el ganador debería ser el mismo
**documento** en ambas corridas (solo cambia su etiqueta A/B). Definimos:

$$\text{Position Bias Rate} = \frac{\#\{\text{pares donde el veredicto se invierte con el orden}\}}{\#\{\text{pares evaluados}\}}$$

Un juez perfecto tendría Position Bias Rate $= 0$. En la práctica, hasta jueces
potentes muestran algo de sesgo — mitigación estándar: **evaluar siempre en ambos
órdenes y promediar**, o descartar el veredicto si no hay consenso.

### Con Valentina

Si Valentina, viendo los mismos 2 dibujos, dijera *"el de la izquierda"* sin importar
CUÁL dibujo esté a la izquierda — eso es sesgo de posición puro: no está juzgando
los dibujos, está juzgando la posición.

In [ ]:
def check_position_bias(question, answer_x, answer_y, context, label_x="X", label_y="Y"):
    v1 = judge_pairwise(question, answer_x, answer_y, context)   # X=A, Y=B
    v2 = judge_pairwise(question, answer_y, answer_x, context)   # Y=A, X=B (orden invertido)

    winner_1 = {"A": label_x, "B": label_y, "Tie": "Tie", None: "?"}[v1]
    winner_2 = {"A": label_y, "B": label_x, "Tie": "Tie", None: "?"}[v2]  # ¡OJO! A/B invertidos

    consistent = winner_1 == winner_2
    return consistent, winner_1, winner_2

print("🔬 Test de sesgo de posición sobre varios pares:\n")
pairs_to_test = [
    ("✅ buena", "❌ alucinada"),
    ("✅ buena", "❌ irrelevante"),
    ("❌ alucinada", "❌ evasiva"),
]

flips = 0
for label_x, label_y in pairs_to_test:
    consistent, w1, w2 = check_position_bias(
        question, cases[label_x], cases[label_y], context, label_x, label_y
    )
    flips += not consistent
    icon = "✅ consistente" if consistent else "🚨 SE INVIRTIÓ"
    print(f"{label_x} vs {label_y}:  orden 1 → gana '{w1}'  |  orden 2 → gana '{w2}'   [{icon}]")

rate = flips / len(pairs_to_test)
print(f"\n📊 Position Bias Rate = {flips}/{len(pairs_to_test)} = {rate:.2f}")
print("""
💡 Con un modelo tan chico como Gemma 3 1B corriendo en modo determinista
   (do_sample=False), es esperable ver bastante inconsistencia — parte es
   sesgo de posición real, parte es simplemente un juez débil. Con un modelo
   más grande (o GPT-4-class), esperarías una tasa bastante menor.
""")

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🔬 Test de sesgo de posición sobre varios pares:



[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ buena vs ❌ alucinada:  orden 1 → gana '✅ buena'  |  orden 2 → gana '✅ buena'   [✅ consistente]


[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ buena vs ❌ irrelevante:  orden 1 → gana '✅ buena'  |  orden 2 → gana '✅ buena'   [✅ consistente]


[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


❌ alucinada vs ❌ evasiva:  orden 1 → gana '❌ alucinada'  |  orden 2 → gana '❌ alucinada'   [✅ consistente]

📊 Position Bias Rate = 0/3 = 0.00

💡 Con un modelo tan chico como Gemma 3 1B corriendo en modo determinista
   (do_sample=False), es esperable ver bastante inconsistencia — parte es
   sesgo de posición real, parte es simplemente un juez débil. Con un modelo
   más grande (o GPT-4-class), esperarías una tasa bastante menor.



---
# PARTE 5: Sesgo de verbosidad (*verbosity bias*)

## Concepto

Los LLM-jueces tienden a preferir respuestas **más largas**, incluso cuando el
contenido extra no aporta información nueva — confunden "detallado" con "correcto".
Esto es especialmente relevante para nosotros: recuerda que en la Clase 1 notamos
que un modelo de chat como Gemma tiende a ser más verboso que `flan-t5`. Si
además el JUEZ premia la verbosidad, se genera un incentivo perverso: el sistema
aprende a "hablar más" en vez de "responder mejor".

### Con Valentina

Dos dibujos que cumplen igual de bien la consigna — pero uno tiene muchos más
colores y garabatos de relleno. Si Valentina prefiere el más "ocupado" solo por
tener más de todo, no está evaluando calidad — está confundiendo cantidad con calidad.

In [ ]:
concisa = cases["✅ buena"]   # "On the Free plan, files can be up to 2GB each. Deleted files stay in the Trash for 30 days before being permanently removed."

# Mismo contenido exacto, pero envuelto en relleno verboso que no añade información nueva
verbosa = (
    "That's a great question, and I'm happy to walk you through it in detail! "
    "To begin with, it's worth noting that CloudBox offers a range of thoughtfully "
    "designed plans to suit different needs. Specifically, on the Free plan, files "
    "can be up to 2GB each. Furthermore, in terms of file recovery, it's important "
    "to mention that deleted files stay in the Trash for 30 days before being "
    "permanently removed, giving you plenty of time to restore them if needed."
)

print(f"Concisa ({len(concisa.split())} palabras): {concisa}\n")
print(f"Verbosa ({len(verbosa.split())} palabras): {verbosa[:120]}...\n")

verdict = judge_pairwise(question, concisa, verbosa, context, max_token=200)
verdict_swapped = judge_pairwise(question, verbosa, concisa, context, max_token=200)

print(f"judge_pairwise(A=concisa, B=verbosa)  → {verdict}")
print(f"judge_pairwise(A=verbosa, B=concisa)  → {verdict_swapped}")
print("""
❓ Si el juez prefirió la verbosa en AMBOS órdenes (controlando así por sesgo de
   posición), eso es evidencia de sesgo de verbosidad — misma información,
   más palabras, mejor score. ¿Es lo que observaste?
""")

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Concisa (24 palabras): On the Free plan, files can be up to 2GB each. Deleted files stay in the Trash for 30 days before being permanently removed.

Verbosa (79 palabras): That's a great question, and I'm happy to walk you through it in detail! To begin with, it's worth noting that CloudBox ...



[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


judge_pairwise(A=concisa, B=verbosa)  → A
judge_pairwise(A=verbosa, B=concisa)  → A

❓ Si el juez prefirió la verbosa en AMBOS órdenes (controlando así por sesgo de
   posición), eso es evidencia de sesgo de verbosidad — misma información,
   más palabras, mejor score. ¿Es lo que observaste?



---
# PARTE 6: Sesgo de auto-preferencia (*self-preference bias*)

## Concepto

Cuando el juez y el generador son el **mismo modelo** (o de la misma familia), el juez
tiende a preferir el estilo de escritura que él mismo produciría — investigaciones
recientes muestran que modelos como GPT-4 tienden a puntuar más alto las respuestas
generadas por sí mismos que las de otros modelos, incluso cuando la calidad es
comparable. Esto es exactamente nuestra situación hoy: **Gemma juzga respuestas,
y muchas veces esas respuestas fueron generadas por Gemma** (Clase 3, y el resto
de este curso).

### Con Valentina

Valentina dibuja gatos en su tiempo libre. Sin darse cuenta, cuando juzga el
concurso, les da mejor puntaje a los dibujos de gatos que a los de perros —
aunque el perro esté igual de bien dibujado. No es que sea injusta a propósito:
prefiere lo que se parece a su propio estilo.

### Por qué esto es difícil de demostrar con un solo modelo local

Para medir esto de verdad necesitas **dos generadores distintos** (ej. Gemma vs
GPT-4 vs Claude) y un juez, y comparar si el juez favorece sistemáticamente al
modelo de su propia familia. Con un solo modelo corriendo en Colab no podemos
aislar el efecto limpiamente — pero sí podemos dejar el experimento armado para
que lo extiendas.

In [ ]:
# Una respuesta "en el estilo de Gemma" (generada por el modelo)
prompt_gen = f"Answer based on the context, in one short sentence.\n\nContext: {context}\n\nQuestion: {question}\n\nAnswer:"
respuesta_gemma = ask_llm(prompt_gen, max_new_tokens=60)

# Una respuesta escrita "a mano" por el instructor, con contenido equivalente pero estilo distinto
respuesta_instructor = "Free tier: 2GB max per file. Trash retention: 30 days."

print(f"🤖 Respuesta de Gemma:      {respuesta_gemma}")
print(f"✍️  Respuesta del instructor: {respuesta_instructor}\n")

verdict = judge_pairwise(question, respuesta_gemma, respuesta_instructor, context, max_token=200)
verdict_swapped = judge_pairwise(question, respuesta_instructor, respuesta_gemma, context, max_token=200)
print(f"judge_pairwise(A=Gemma, B=instructor)  → {verdict}")
print(f"judge_pairwise(A=instructor, B=Gemma)  → {verdict_swapped}")

print("""
💡 Esto NO aísla self-preference bias de forma limpia (podría ganar simplemente
   porque es más completa, no por "sonar como Gemma"). Es un punto de partida —
   el Ejercicio 5 de esta clase te pide extenderlo con un segundo modelo real.
""")

[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Respuesta de Gemma:      Free plan users can upload files up to 2GB each, and deleted files stay in the Trash for 30 days.
✍️  Respuesta del instructor: Free tier: 2GB max per file. Trash retention: 30 days.



[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


judge_pairwise(A=Gemma, B=instructor)  → A
judge_pairwise(A=instructor, B=Gemma)  → A

💡 Esto NO aísla self-preference bias de forma limpia (podría ganar simplemente
   porque es más completa, no por "sonar como Gemma"). Es un punto de partida —
   el Ejercicio 5 de esta clase te pide extenderlo con un segundo modelo real.



---
# PARTE 7: Calibración contra evaluación humana

Todo lo anterior asumió que el juez "más o menos funciona". Pero, ¿cómo lo sabemos
con números? **Calibrando** sus veredictos contra etiquetas puestas por un humano.

## Fundamento matemático: Cohen's Kappa

La forma ingenua de medir acuerdo es el **porcentaje de acuerdo simple**:

$$p_o = \frac{\#\{\text{casos donde humano y juez coinciden}\}}{N}$$

**Problema**: si el 90% de tus datos son "buenos", dos evaluadores que solo dijeran
"bueno" siempre acordarían 90% del tiempo **por pura casualidad**, sin haber mirado
nada. $p_o$ no distingue acuerdo real de acuerdo por azar.

**Cohen's Kappa** corrige por esto, restando el acuerdo esperado al azar $p_e$:

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

$$p_e = \sum_{c \,\in\, \text{categorías}} \left(\frac{n^{humano}_c}{N}\right) \left(\frac{n^{juez}_c}{N}\right)$$

$p_e$ es la probabilidad de que humano y juez coincidan **por azar**, dado cuántas
veces cada uno usó cada categoría (sus "marginales").

### Interpretación (escala de Landis & Koch)

| $\kappa$ | Interpretación |
|---|---|
| < 0 | peor que el azar |
| 0.00 – 0.20 | acuerdo insignificante |
| 0.21 – 0.40 | acuerdo débil |
| 0.41 – 0.60 | acuerdo moderado |
| 0.61 – 0.80 | acuerdo sustancial |
| 0.81 – 1.00 | acuerdo casi perfecto |

### Con Valentina — ejemplo a mano

Valentina calificó 10 dibujos como "Bueno" o "Malo". Comparamos contra la profesora
de arte (nuestro "humano de referencia"):

| | Profesora: Bueno | Profesora: Malo |
|---|:---:|:---:|
| **Valentina: Bueno** | 6 | 1 |
| **Valentina: Malo** | 1 | 2 |

- $p_o = \frac{6+2}{10} = 0.80$ (acuerdo simple: 80%... ¡suena bien!)
- Marginales: Profesora dijo "Bueno" 7 veces, "Malo" 3; Valentina igual: 7 y 3
- $p_e = \left(\frac{7}{10}\cdot\frac{7}{10}\right) + \left(\frac{3}{10}\cdot\frac{3}{10}\right) = 0.49 + 0.09 = 0.58$
- $\kappa = \frac{0.80 - 0.58}{1 - 0.58} = \frac{0.22}{0.42} \approx 0.52$ → **acuerdo moderado**

Nota cómo el 80% "de acuerdo simple" que sonaba muy bien se convierte en un κ de
solo 0.52 una vez que descontamos la suerte — esa es exactamente la corrección que
Cohen's Kappa aporta.

In [ ]:
def cohens_kappa(human_labels, judge_labels):
    n = len(human_labels)
    categories = sorted(set(human_labels) | set(judge_labels))
    po = sum(h == j for h, j in zip(human_labels, judge_labels)) / n
    human_counts = Counter(human_labels)
    judge_counts = Counter(judge_labels)
    pe = sum((human_counts[c] / n) * (judge_counts[c] / n) for c in categories)
    if pe == 1:
        return 1.0
    return (po - pe) / (1 - pe)

# Verificación con el ejemplo a mano de Valentina
human_example = ["Bueno"]*6 + ["Bueno"]*1 + ["Malo"]*1 + ["Malo"]*2
judge_example = ["Bueno"]*6 + ["Malo"]*1 + ["Bueno"]*1 + ["Malo"]*2

po = sum(h == j for h, j in zip(human_example, judge_example)) / len(human_example)
kappa = cohens_kappa(human_example, judge_example)
print(f"p_o (acuerdo simple) = {po:.2f}  (esperado 0.80)")
print(f"kappa                = {kappa:.4f}  (esperado ≈0.5238)")

p_o (acuerdo simple) = 0.80  (esperado 0.80)
kappa                = 0.5238  (esperado ≈0.5238)


## Calibración real: Valentina (humano) vs Gemma (juez), sobre CloudBox

Ahora hacemos el ejercicio real: tú (el instructor / "el humano") ya calificaste
8 respuestas de CloudBox. Comparamos esas etiquetas contra lo que dice
`judge_pointwise`, categorizando ambos en "Aceptable" (score ≥ 4) vs "No aceptable"
(score < 4) para poder aplicar el mismo Cohen's Kappa categórico.

In [ ]:
# 8 respuestas de ejemplo con su etiqueta HUMANA ya decidida (simulando anotación manual)
calibration_set = [
    {"question": question, "answer": cases["✅ buena"], "human_label": "Aceptable"},
    {"question": question, "answer": cases["❌ alucinada"], "human_label": "No aceptable"},
    {"question": question, "answer": cases["❌ irrelevante"], "human_label": "No aceptable"},
    {"question": question, "answer": cases["❌ evasiva"], "human_label": "No aceptable"},
    {"question": "How much storage does the Pro plan include?",
     "answer": "The Pro plan includes 500GB of storage for $9 per month.",
     "human_label": "Aceptable"},
    {"question": "How much storage does the Pro plan include?",
     "answer": "The Pro plan includes unlimited storage for free.",
     "human_label": "No aceptable"},
    {"question": "What kind of support do Team plan users get?",
     "answer": "Team plan users get priority live chat support.",
     "human_label": "Aceptable"},
    {"question": "What kind of support do Team plan users get?",
     "answer": "All plans get the exact same support, there is no difference.",
     "human_label": "No aceptable"},
]

human_labels, judge_labels, judge_scores = [], [], []

print("🔬 Calibrando el juez contra las etiquetas humanas:\n")
for item in calibration_set:
    score, _ = judge_pointwise(item["question"], item["answer"], context)
    judge_label = "Aceptable" if (score is not None and score >= 4) else "No aceptable"

    human_labels.append(item["human_label"])
    judge_labels.append(judge_label)
    judge_scores.append(score)

    match = "✅" if judge_label == item["human_label"] else "❌"
    print(f"{match} humano={item['human_label']:<13} juez={judge_label:<13} (score={score})  '{item['answer'][:50]}...'")

po = sum(h == j for h, j in zip(human_labels, judge_labels)) / len(human_labels)
kappa = cohens_kappa(human_labels, judge_labels)

print(f"\n📊 Acuerdo simple (p_o) = {po:.2f}")
print(f"📊 Cohen's Kappa        = {kappa:.2f}")
print("""
❓ Con κ bajo (0-0.4) el juez no es confiable todavía para decisiones
   importantes sin supervisión — hace falta mejor rúbrica, más ejemplos, o un
   modelo más grande. Con κ alto (>0.6), podrías empezar a confiar en el juez
   para monitoreo continuo, revisando manualmente solo una muestra.
""")

[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🔬 Calibrando el juez contra las etiquetas humanas:



[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ humano=Aceptable     juez=Aceptable     (score=5)  'On the Free plan, files can be up to 2GB each. Del...'


[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

❌ humano=Aceptable     juez=No aceptable  (score=2)  'The Pro plan includes 500GB of storage for $9 per ...'


[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ humano=No aceptable  juez=No aceptable  (score=2)  'The Pro plan includes unlimited storage for free....'


[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


❌ humano=Aceptable     juez=No aceptable  (score=2)  'Team plan users get priority live chat support....'
✅ humano=No aceptable  juez=No aceptable  (score=2)  'All plans get the exact same support, there is no ...'

📊 Acuerdo simple (p_o) = 0.75
📊 Cohen's Kappa        = 0.38

❓ Con κ bajo (0-0.4) el juez no es confiable todavía para decisiones
   importantes sin supervisión — hace falta mejor rúbrica, más ejemplos, o un
   modelo más grande. Con κ alto (>0.6), podrías empezar a confiar en el juez
   para monitoreo continuo, revisando manualmente solo una muestra.



## La librería real: `sklearn` ya la implementa

En producción no reimplementas Cohen's Kappa a mano — usas la función ya probada:

```python
from sklearn.metrics import cohen_kappa_score
kappa = cohen_kappa_score(human_labels, judge_labels)
```

Misma fórmula que programaste arriba — con mejor manejo de casos borde (por ejemplo,
cuando todas las etiquetas son iguales).

In [ ]:
from sklearn.metrics import cohen_kappa_score

kappa_sklearn = cohen_kappa_score(human_labels, judge_labels)
kappa_manual = cohens_kappa(human_labels, judge_labels)
print(f"Nuestra implementación : {kappa_manual:.4f}")
print(f"sklearn                : {kappa_sklearn:.4f}")
print("✅ Coinciden" if abs(kappa_manual - kappa_sklearn) < 1e-9 else "⚠️ No coinciden — revisar")

Nuestra implementación : 0.3846
sklearn                : 0.3846
✅ Coinciden


---
# PARTE 8: Herramientas de la industria

No reimplementas todo esto desde cero en un proyecto real. Algunas herramientas del
ecosistema:

- **promptfoo**: framework open-source para testear prompts con *assertions*
  (incluye LLM-graded assertions — exactamente el patrón `judge_pointwise` que
  construiste hoy, pero integrado a un pipeline de CI/CD). Lo vamos a usar en el
  Bloque 3 como *gate* de regression testing.
- **MT-Bench / Chatbot Arena**: metodologías de referencia de la industria para
  evaluar LLMs con jueces (MT-Bench) y con preferencia humana pairwise a escala
  (Arena) — la inspiración de la Parte 3 de hoy.
- **LangSmith / Langfuse**: además de tracing (Bloque 2), incluyen "evaluators"
  configurables que son, en esencia, `judge_pointwise` con UI y almacenamiento
  histórico de resultados.

La idea que se repite en todas: la infraestructura cambia, pero **rúbrica + juez +
calibración periódica contra humanos** es el patrón de fondo en todas ellas.


---
# Ejercicios Propuestos

1. **Rúbrica alternativa**: diseña una rúbrica para evaluar *tono* (¿la respuesta
   suena profesional y amigable, o cortante?) en vez de groundedness. Pruébala
   sobre los 4 casos de CloudBox. ¿El ranking cambia respecto a la Parte 2?

2. **Posición, a fondo**: repite el experimento de la Parte 4 con los 6 pares
   posibles entre los 4 casos (no solo 3). ¿La tasa de sesgo de posición se
   mantiene estable o cambia mucho con más datos?

3. **Verbosidad al extremo**: crea una versión de la respuesta "evasiva" pero
   artificialmente alargada con relleno (sin agregar información real). ¿Puede
   el sesgo de verbosidad hacer que una respuesta evasiva le gane a una alucinada?

4. **Kappa ponderado**: investiga *weighted Cohen's Kappa* (con pesos lineales o
   cuadráticos) — útil cuando las categorías tienen orden (como nuestros scores
   1-5) y un desacuerdo de "4 vs 5" debería pesar menos que uno de "1 vs 5".
   Impleméntalo y compáralo con el kappa sin ponderar de la Parte 7.

5. **Self-preference real**: si tienes acceso a la API de otro modelo (OpenAI,
   Anthropic, etc.), genera las mismas 8 respuestas del set de calibración con
   ese otro modelo y compara si Gemma-juez las puntúa distinto que a las suyas
   propias, controlando por calidad.

6. **Ensemble de jueces**: en vez de un solo juez, pide 3 veredictos con
   `do_sample=True` y quédate con el veredicto mayoritario (*self-consistency*).
   ¿Mejora la tasa de sesgo de posición de la Parte 4?


---
# Conclusión

### Sesgos de LLM-as-a-judge, de un vistazo

| Sesgo | Qué pasa | Cómo mitigarlo |
|---|---|---|
| **Posición** | prefiere sistemáticamente A o B | evaluar en ambos órdenes y promediar/descartar inconsistentes |
| **Verbosidad** | confunde "más largo" con "mejor" | normalizar longitud en el prompt, o pedir explícitamente penalizar el relleno |
| **Auto-preferencia** | prefiere el estilo de su propia familia de modelos | usar un juez de familia distinta al generador, cuando sea posible |
| **Rúbrica vaga** | alta varianza entre corridas | anclas explícitas por cada punto de la escala |

### Ideas clave
1. **LLM-as-a-judge no es magia** — es un patrón (rúbrica + prompt + parsing) con
   los mismos riesgos de cualquier LLM: hay que diseñarlo con cuidado y auditarlo
2. **Pairwise > Pointwise** cuando lo que necesitas es decidir entre dos versiones;
   pointwise cuando necesitas una serie temporal para monitorear
3. Los sesgos son **medibles**, no solo teóricos — hoy los cuantificaste con código
4. **Cohen's Kappa** separa acuerdo real de acuerdo por azar — un 80% de acuerdo
   simple puede ser un κ mediocre
5. Ningún juez reemplaza la calibración periódica contra humanos — es el ancla
   que evita que todo el sistema de evaluación se desvíe silenciosamente
